# Demo 02: GPU K-Means Clustering

**Learning objectives:**
- Understand Lloyd's k-means algorithm and how it maps to GPU parallelism
- Run the NumPy CPU baseline interactively and observe its performance
- See how custom CUDA kernels replace the two key Lloyd's operations
- Compare CPU vs GPU timing and understand where the speedup comes from

> **Note:** Run cells top-to-bottom. Restarting the kernel resets all state.
> GPU cells are guarded with `if GPU_AVAILABLE:` and are skipped automatically
> on CPU-only machines.

In [ ]:
# GPU detection — always runs
import subprocess
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    GPU_AVAILABLE = True
    print("GPU detected — all cells will run.")
except (FileNotFoundError, subprocess.CalledProcessError):
    GPU_AVAILABLE = False
    print("No GPU detected — GPU cells will be skipped (CPU baseline still works.)")

## K-Means Algorithm

K-means partitions `N` samples into `k` clusters by iterating two steps until convergence:

1. **Assignment step** — assign each sample to its nearest centroid
   (squared Euclidean distance)
2. **Update step** — recompute each centroid as the mean of its assigned samples

### GPU mapping

Both steps are embarrassingly parallel over samples:

| CPU (NumPy) | GPU (CUDA kernel) |
|------------|--------------------|
| `np.argmin(dists, axis=1)` | `assign_labels` kernel — one thread per sample |
| `X[mask].mean(axis=0)` | `accumulate_centroids` kernel — `atomicAdd` per feature |

The GPU version scales well because assignment is independent across samples
and `atomicAdd` allows concurrent accumulation without synchronization overhead
on large batches.

In [ ]:
# Parameter setup — user-editable
import sys
sys.path.insert(0, "..")
import numpy as np

N_SAMPLES = 5_000    # number of data points; try 10_000 or 50_000
N_FEATURES = 16      # dimensionality; try 2, 8, 32
K = 8                # number of clusters; try 4, 16, 32
SEED = 42

rng = np.random.default_rng(SEED)
X = rng.standard_normal((N_SAMPLES, N_FEATURES)).astype(np.float32)
print(f"Dataset shape : {X.shape}")
print(f"dtype         : {X.dtype}")
print(f"Memory        : {X.nbytes / 1024:.1f} KB")
print(f"K             : {K} clusters")

## CPU Baseline (NumPy)

The CPU baseline implements Lloyd's algorithm with pure NumPy.
The assignment step uses broadcasting: `(X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2`
which creates an `(N, k, features)` tensor — correct but memory-intensive.

Run this cell to measure CPU k-means time on your machine.

In [ ]:
# CPU baseline — always runs (no GPU required)
import time
import importlib.util

# Load cpu_kmeans directly from file to avoid Python import name collision
# (directory name '02_kmeans' starts with a digit, which is not valid in Python identifiers)
_spec = importlib.util.spec_from_file_location(
    "cpu_kmeans", "../demos/02_kmeans/cpu_kmeans.py"
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
kmeans_cpu = _mod.kmeans_cpu

print(f"Running CPU k-means: N={N_SAMPLES:,}, features={N_FEATURES}, k={K} ...")
t0 = time.perf_counter()
cpu_centroids, cpu_labels, cpu_inertia = kmeans_cpu(X, k=K, seed=SEED)
cpu_time = time.perf_counter() - t0

print(f"\nCPU k-means results:")
print(f"  Time     : {cpu_time * 1000:.1f} ms")
print(f"  Inertia  : {cpu_inertia:,.1f}")
print(f"  Centroids: {cpu_centroids.shape}")

# Cluster size distribution
unique, counts = np.unique(cpu_labels, return_counts=True)
print(f"\nCluster sizes: min={counts.min()}, max={counts.max()}, mean={counts.mean():.0f}")

## GPU K-Means Design

The GPU version replaces the two NumPy operations with custom CUDA kernels:

### Kernel 1: `assign_labels`
```c
__global__ void assign_labels(
    const float* X, const float* centroids, int* labels,
    int n_samples, int n_features, int k
) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    // each thread: compute distance to all k centroids, store argmin
}
```

### Kernel 2: `accumulate_centroids`
```c
__global__ void accumulate_centroids(
    const float* X, const int* labels, float* new_centroids, int* counts, ...
) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    // each thread: atomicAdd to its cluster's sum and count
}
```

**Block size**: 256 threads/block — good occupancy for register-light kernels.
See `01_core_apis.ipynb` for occupancy analysis.

In [ ]:
# REQUIRES GPU
if GPU_AVAILABLE:
    import importlib.util as _ilu

    _gspec = _ilu.spec_from_file_location(
        "gpu_kmeans", "../demos/02_kmeans/gpu_kmeans.py"
    )
    _gmod = _ilu.module_from_spec(_gspec)
    _gspec.loader.exec_module(_gmod)
    kmeans_gpu = _gmod.kmeans_gpu

    print(f"Running GPU k-means: N={N_SAMPLES:,}, features={N_FEATURES}, k={K} ...")
    t0 = time.perf_counter()
    gpu_centroids, gpu_labels, gpu_inertia = kmeans_gpu(X, k=K, seed=SEED)
    gpu_time = time.perf_counter() - t0

    print(f"\nGPU k-means results:")
    print(f"  Time     : {gpu_time * 1000:.1f} ms")
    print(f"  Inertia  : {gpu_inertia:,.1f}")
    print(f"  Centroids: {gpu_centroids.shape}")
else:
    print("GPU not available — skipping GPU k-means.")
    print(f"Expected speedup for N={N_SAMPLES:,}, k={K}: ~10-50x depending on GPU.")
    gpu_time = None
    gpu_inertia = None

In [ ]:
# Speedup comparison — always runs, GPU section conditional
print("=" * 50)
print("CPU vs GPU K-Means Comparison")
print("=" * 50)
print(f"Dataset        : N={N_SAMPLES:,}, features={N_FEATURES}, k={K}")
print(f"CPU time       : {cpu_time * 1000:.1f} ms")

if GPU_AVAILABLE and gpu_time is not None:
    speedup = cpu_time / gpu_time
    print(f"GPU time       : {gpu_time * 1000:.1f} ms")
    print(f"Speedup        : {speedup:.1f}x")
    print(f"CPU inertia    : {cpu_inertia:,.1f}")
    print(f"GPU inertia    : {gpu_inertia:,.1f}")
    inertia_diff = abs(cpu_inertia - gpu_inertia) / cpu_inertia * 100
    print(f"Inertia diff   : {inertia_diff:.2f}% (float32 precision)")
else:
    print("GPU time       : N/A (no GPU detected)")
    print("Speedup        : N/A")
    print("")
    print("Install CUDA Toolkit 12.x and run:")
    print("  pip install cuda-python")
    print("  python demos/02_kmeans/main.py")

## Understanding the Speedup

### Why GPU k-means is faster

The **assignment step** is the bottleneck at large N. Each sample independently
computes its distance to all `k` centroids — O(N × k × features) operations.
With 5,000 samples and k=8, that is 640,000 independent distance computations
per iteration. On a GPU with thousands of CUDA cores, most of these run in
parallel.

### Roofline perspective

K-means assignment reads each sample once and reads all k centroids
for every sample. With small k, the centroid data fits in L2 cache,
and arithmetic intensity rises:

```
FLOPs per sample ≈ 2 × k × features   (subtract + square per feature × k centroids)
Bytes per sample ≈ features × 4 bytes  (one read from global memory)
Arithmetic intensity ≈ 2 × k = 16 FLOP/byte  (for k=8)
```

At 16 FLOP/byte, the kernel approaches the ridge point of the A100
(~9.6 FLOP/byte), making it lightly compute-bound — a much better
use of GPU FP32 throughput than vector-add (~0.08 FLOP/byte).

### Why speedup grows with N

GPU kernels have a fixed launch overhead (~10-50 µs). For small N,
this overhead dominates. For large N (100,000+), the parallelism
advantage compounds and speedups of 20-100x are typical.

## Summary

In this notebook we explored:

1. **Lloyd's k-means algorithm** — assignment and update steps, and why
   they map naturally to GPU parallelism
2. **CPU NumPy baseline** — ran `kmeans_cpu()` and measured wall-clock time
3. **GPU kernel design** — `assign_labels` and `accumulate_centroids` CUDA kernels
4. **Performance analysis** — roofline perspective on why k-means is
   faster on GPUs than vector-add (higher arithmetic intensity)

**Experiments to try:**
- Increase `N_SAMPLES` to 50,000 or 100,000 and observe how speedup scales
- Vary `K` from 4 to 64 — how does increasing k affect CPU vs GPU time?
- Vary `N_FEATURES` — higher dimensionality increases arithmetic intensity

**Next steps:**
- `demos/02_kmeans/main.py` — full CLI demo with timing
- `demos/03_pca/` — GPU Principal Component Analysis
- `benchmarks/run_all.py` — comparison across all 8 demos